# server.py: The Calculator MCP server

**`server.py` is a small MCP server that offers one tool, `add`.** It can talk over stdio, Streamable HTTP or SSE, and you choose which one with the word you type after `python server.py`:

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 you type                 transport         clients connect to
 --------                 ---------         ------------------
 python server.py         Streamable HTTP   http://127.0.0.1:8050/mcp
 python server.py stdio   stdio             (the client starts it)
 python server.py sse     SSE (legacy)      http://127.0.0.1:8050/sse
</pre>

This notebook reads the file from top to bottom. It only imports and inspects `server.py`, and never starts the server. Read it before the client notebooks: `client-stdio.ipynb`, then `client-streamable-http.ipynb`.

## The whole file

<div style="font-size: 0.85em">

```python
import sys

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Calculator", port=8050)


@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers together"""
    return a + b


if __name__ == "__main__":
    transport = "streamable-http"
    if len(sys.argv) > 1:
        transport = sys.argv[1]

    print("Calculator server starting with transport:", transport, file=sys.stderr)
    mcp.run(transport=transport)
```

</div>

## 1. Creating the server

`FastMCP` is the server class in MCP's official Python SDK (the `mcp` package). `FastMCP("Calculator", port=8050)` creates a server called Calculator:

- **The name, `"Calculator"`:** Clients see it in the handshake reply, under `serverInfo`.
- **`port=8050`:** The door number the two HTTP transports listen at. stdio doesn't use a port.
- **No host given:** FastMCP listens on `127.0.0.1` by default. That address means "this computer", so only programs on your own machine can reach the server. It's the safe default for a local server.

## 2. The tool

`@mcp.tool()` is a decorator: The line above a function that hands the function to something else. Here it registers `add` with the server as a tool. FastMCP builds the tool's description from the docstring, and its input schema from the type hints (`a: int, b: int`). That's the same kind of JSON schema you wrote by hand for `get_weather` in `3-tools.py`.

To see what FastMCP built, import the file and ask the server for its tool list directly, with no client and no transport:

- **`import server`:** Runs `server.py` as a module. It creates `mcp` and registers `add`, but doesn't start the server (section 4 shows why).
- **`server.mcp.list_tools()`:** FastMCP's own method. It returns the registered tools.
- **`await`:** Waits for an async method to finish. A notebook lets you use it directly in a cell.
- **`model_dump_json(indent=2, exclude_none=True)`:** Prints the tool as indented JSON and leaves out empty fields.

In [1]:
import server

tools = await server.mcp.list_tools()
print(tools[0].model_dump_json(indent=2, exclude_none=True))

{
  "name": "add",
  "description": "Add two numbers together",
  "inputSchema": {
    "properties": {
      "a": {
        "title": "A",
        "type": "integer"
      },
      "b": {
        "title": "B",
        "type": "integer"
      }
    },
    "required": [
      "a",
      "b"
    ],
    "title": "addArguments",
    "type": "object"
  },
  "outputSchema": {
    "properties": {
      "result": {
        "title": "Result",
        "type": "integer"
      }
    },
    "required": [
      "result"
    ],
    "title": "addOutput",
    "type": "object"
  }
}


- **`description`:** The docstring, word for word.
- **`inputSchema`:** One `integer` property per argument, and both are `required`. This is what the model reads to know what to send.
- **`outputSchema`:** Built from `-> int`. FastMCP wraps a plain return value as `{"result": ...}`.

## 3. Choosing the transport

`sys` is Python's built-in module for information about the running program. `sys.argv` is the list of words on the command line, starting with the file name:

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 you type                  sys.argv
 python server.py          ["server.py"]
 python server.py stdio    ["server.py", "stdio"]
</pre>

So `len(sys.argv) > 1` means "a word was typed after the file name", and `sys.argv[1]` is that word. With no word, `transport` stays `"streamable-http"`. A word FastMCP doesn't know stops the server with `ValueError: Unknown transport: foo`.

## 4. `if __name__ == "__main__":`

`__name__` is a variable Python sets in every file. It's `"__main__"` when you run the file directly (`python server.py`), and the file's own name when another file imports it:

In [2]:
print(server.__name__)

server


So the block under the `if` runs for `python server.py ...` and is skipped for `import server`. That's why section 2 could import the file without starting the server.

## 5. Why the startup message goes to stderr

Every program has two output streams. **stdout** is where `print()` writes by default. **stderr** is a second stream, meant for messages and errors.

Over stdio, stdout carries the JSON messages to the client, so an ordinary line of text there would reach the client as a broken message. `print(..., file=sys.stderr)` sends the startup line to stderr instead. FastMCP writes its own log lines ("Processing request of type ...") to stderr too.

## 6. Starting the server

`mcp.run(transport=transport)` starts the server and keeps it running. An HTTP server runs until you press `Ctrl+C`. A stdio server runs until its client closes the connection.

To start the HTTP server, open a terminal in this folder:

<div style="font-size: 0.85em">

```bash
conda activate MCP_env
python server.py
```

</div>

It prints:

<div style="font-size: 0.85em">

```text
Calculator server starting with transport: streamable-http
INFO:     Started server process [24692]
INFO:     Waiting for application startup.
StreamableHTTP session manager started
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8050 (Press CTRL+C to quit)
```

</div>

The first line is the file's own `print`. The `INFO:` lines come from uvicorn, the small web server FastMCP uses for HTTP. You never start a stdio server yourself: The client does it, in `client-stdio.ipynb`.

## Say it out loud

> "My server is a FastMCP server with one tool, `add`. The `@mcp.tool()` decorator registers it, and FastMCP turns the type hints into the input schema and the docstring into the description. I pick the transport when I start it, from stdio, Streamable HTTP or SSE. It listens on 127.0.0.1 by default, and it logs to stderr, because over stdio, stdout is the protocol channel."